# تدريب Zazo Brain من Cloudflare KV باستخدام Transformers + PEFT

دفتر Colab كامل لجلب سجلات JSONL من Cloudflare KV، وتحويلها إلى محادثات، ثم تدريب **LoRA غير مكمّم** على `google/gemma-2b-it` باستخدام Transformers وPEFT وTRL. يحفظ المحول في Google Drive ويمكن رفع الملفين من واجهة Zazo إلى Cloudflare Workers AI.

**يلزم GPU فعليًا** (مثل T4/L4/A100 إن توفر). هذا الدفتر لا يعمل على CPU أو TPU بصيغته الحالية. اقبل شروط Gemma على Hugging Face قبل تنزيل النموذج. لا تضع Cloudflare API Token في الدفتر؛ سيُطلب عند سحب بيانات KV.


## 1. تثبيت الحزم

In [ ]:
# ثبّت المكتبات، ثم أعد تشغيل الجلسة من Runtime → Restart session عند طلب ذلك.
%pip install -q -U transformers datasets peft trl accelerate huggingface_hub


## 2. تحقق من GPU

In [ ]:
import torch, sys
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA متاحة:", torch.cuda.is_available())
print("عدد وحدات GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("لا تتابع إلى تدريب النموذج حتى تتصل جلسة Colab بوحدة GPU.")


## 3. إعدادات الحساب والموديل

معرف الحساب ومساحة KV مضمّنان حسب المعرفات التي زودتني بها. أدخل Cloudflare Token عند تشغيل الخلية في الحقل الآمن. الحساب الذي يفتح Google Drive يُختار عند تنفيذ mount؛ تحقق من الحساب الظاهر في نافذة Google قبل السماح بالوصول.

In [ ]:
import os, getpass, json, time, urllib.parse, requests

CF_ACCOUNT_ID = "f116fb028284f528f1eeb6f487088acf"
CF_KV_NAMESPACE_ID = "95f488c06b3349d9aa57f3c2ec4ff20a"
CF_API_TOKEN = getpass.getpass("أدخل Cloudflare API Token (لن يظهر أثناء الكتابة): ")

# حافظ على Gemma 2B instruct كقاعدة متوافقة مع مسار Cloudflare LoRA الذي اخترناه.
# يلزم قبول شروط النموذج على Hugging Face قبل تحميله.
MODEL_NAME = "google/gemma-2b-it"
MAX_SEQ_LENGTH = 1024
TRAINING_PREFIX = "training/"


## 4. الاتصال بـKV وجلب كل المفاتيح مع Pagination

In [ ]:
API_BASE = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/storage/kv/namespaces/{CF_KV_NAMESPACE_ID}"
HEADERS = {"Authorization": f"Bearer {CF_API_TOKEN}"}

def cloudflare_json_get(url, params=None):
    response = requests.get(url, headers=HEADERS, params=params, timeout=60)
    if not response.ok:
        raise RuntimeError(f"Cloudflare API HTTP {response.status_code}: {response.text[:500]}")
    payload = response.json()
    if not payload.get("success", False):
        raise RuntimeError(f"Cloudflare API error: {payload.get('errors', [])}")
    return payload

def list_all_kv_keys(prefix=None, page_size=1000):
    # واجهة Cloudflare KV تستخدم cursor opaque، ونمرره كما ورد دون تغييره.
    url = API_BASE + "/keys"
    cursor = None
    all_keys = []
    seen_cursors = set()
    while True:
        params = {"limit": page_size}
        if prefix:
            params["prefix"] = prefix
        if cursor:
            params["cursor"] = cursor
        payload = cloudflare_json_get(url, params=params)
        all_keys.extend(payload.get("result", []))
        cursor = (payload.get("result_info") or {}).get("cursor")
        if not cursor:
            break
        if cursor in seen_cursors:
            raise RuntimeError("Cloudflare أعاد cursor مكررًا؛ أوقفت pagination لتجنب حلقة لا نهائية.")
        seen_cursors.add(cursor)
    return all_keys

kv_keys = list_all_kv_keys(prefix=TRAINING_PREFIX)
jsonl_keys = [k["name"] for k in kv_keys if k.get("name", "").endswith(".jsonl")]
print("عدد المفاتيح تحت بادئة التدريب:", len(kv_keys))
print("مفاتيح JSONL:", jsonl_keys)


## 5. تنزيل قيم JSONL وتحليلها

تعالج الخلية كل مفتاح JSONL بصورة مستقلة، وتتجاوز القيم الفارغة أو السجلات غير الصالحة مع تقرير موجز، من دون طباعة محتوى بياناتك كاملة.

In [ ]:
def fetch_kv_value(key_name):
    encoded_key = urllib.parse.quote(key_name, safe="")
    url = API_BASE + "/values/" + encoded_key
    response = requests.get(url, headers=HEADERS, timeout=120)
    if response.status_code == 404:
        print("تجاوز مفتاح اختفى أثناء القراءة:", key_name)
        return ""
    if not response.ok:
        raise RuntimeError(f"تعذر قراءة {key_name}: HTTP {response.status_code}: {response.text[:300]}")
    return response.text

raw_records = []
empty_keys = []
parse_errors = []
for key_name in jsonl_keys:
    text = fetch_kv_value(key_name)
    if not text.strip():
        empty_keys.append(key_name)
        continue
    for line_no, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            item = json.loads(line)
            if isinstance(item, dict):
                item["_source_key"] = key_name
                raw_records.append(item)
            else:
                parse_errors.append((key_name, line_no, "السجل ليس JSON object"))
        except json.JSONDecodeError as exc:
            parse_errors.append((key_name, line_no, str(exc)))

print("عدد سجلات JSON المقروءة:", len(raw_records))
print("عدد المفاتيح الفارغة:", len(empty_keys))
print("عدد أسطر JSON غير الصالحة:", len(parse_errors))
if empty_keys:
    print("المفاتيح الفارغة:", empty_keys[:10])
if parse_errors:
    print("أمثلة لأخطاء التحليل (المفتاح، رقم السطر، سبب مختصر):", parse_errors[:5])


## 6. تحويل السجلات إلى محادثات تدريب

In [ ]:
def normalize_record(item):
    # صيغة محادثات جاهزة messages
    messages = item.get("messages")
    if isinstance(messages, list) and messages:
        cleaned = []
        for msg in messages:
            if not isinstance(msg, dict):
                continue
            role = msg.get("role")
            content = msg.get("content")
            if role in {"system", "user", "assistant"} and isinstance(content, str) and content.strip():
                cleaned.append({"role": role, "content": content.strip()})
        if any(m["role"] == "assistant" for m in cleaned):
            return {"messages": cleaned}

    # صيغة Zazo JSONL: instruction/context/response
    instruction = item.get("instruction") or item.get("question") or item.get("prompt") or item.get("input")
    response = item.get("response") or item.get("answer") or item.get("output")
    context = item.get("context") or item.get("system") or ""
    if isinstance(instruction, str) and instruction.strip() and isinstance(response, str) and response.strip():
        user_content = instruction.strip()
        if isinstance(context, str) and context.strip():
            user_content += "\n\nالسياق:\n" + context.strip()
        return {"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": response.strip()},
        ]}
    return None

examples = []
rejected = 0
for record in raw_records:
    normalized = normalize_record(record)
    if normalized:
        examples.append(normalized)
    else:
        rejected += 1

print("أمثلة التدريب الصالحة:", len(examples))
print("سجلات متجاوزة لغياب السؤال/الإجابة:", rejected)
if not examples:
    raise RuntimeError(
        "لم يتم العثور على أمثلة تدريب. تأكد أن مفتاح training/quran-awareness-v1.jsonl "
        "موجود وأن السجلات تحتوي messages أو instruction وresponse. "
        f"المفاتيح المقروءة: {jsonl_keys}"
    )
print("حقول أول مثال:", list(examples[0].keys()))
print("عدد الرسائل في أول مثال:", len(examples[0]["messages"]))


## 7. إعداد Hugging Face وتحميل Gemma

إذا كان النموذج gated، سجّل الدخول إلى Hugging Face بعد قبول شروط النموذج. يمكن استعمال `huggingface_hub.login()` وإدخال الرمز في نافذته الآمنة عند الحاجة.

In [ ]:
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

if not torch.cuda.is_available():
    raise RuntimeError("لا توجد GPU متاحة؛ يلزم GPU لتدريب هذا النموذج. CPU/TPU لا يعملان مع إعداد هذه الخلية.")

# اختياري: سجّل الدخول فقط إذا كان حساب Hugging Face يحتاج رمز وصول.
# login(token=getpass.getpass("Hugging Face token: "))

# تحميل Gemma 2B بوزن FP16 كامل (غير مكمّم) حتى يكون Adapter مناسبًا لمسار Cloudflare.
compute_dtype = torch.float16
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=compute_dtype,
    device_map="auto",
)
model.config.use_cache = False
print("تم تحميل النموذج غير المكمّم على:", next(model.parameters()).device)


## 8. إضافة LoRA وتجهيز نصوص التدريب

In [ ]:
from datasets import Dataset
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def render_messages(example):
    rendered = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": rendered}

train_dataset = Dataset.from_list(examples).map(render_messages)
print("عدد أمثلة التدريب:", len(train_dataset))
print("معاينة أول مثال:\n", train_dataset[0]["text"][:1200])


## 9. التدريب باستخدام TRL SFTTrainer

In [ ]:
from trl import SFTConfig, SFTTrainer

TRAIN_OUTPUT = "/content/zazo_brain_checkpoints"
training_args = SFTConfig(
    output_dir=TRAIN_OUTPUT,
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),
    gradient_checkpointing=True,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    optim="adamw_torch",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)
print("بدء التدريب…")
train_result = trainer.train()
print("اكتمل التدريب.")
print(train_result)


## 10. حفظ محول LoRA في Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# مجلد الناتج النهائي على Drive للحساب الذي اخترته عند mount.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/zazo_brain_lora"
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
model.save_pretrained(DRIVE_OUTPUT_DIR)
tokenizer.save_pretrained(DRIVE_OUTPUT_DIR)
# يطلب Cloudflare وجود model_type داخل adapter_config.json، لذلك نضيفه صراحةً.
adapter_config_path = os.path.join(DRIVE_OUTPUT_DIR, "adapter_config.json")
with open(adapter_config_path, "r", encoding="utf-8") as f:
    adapter_config = json.load(f)
adapter_config["model_type"] = "gemma"
with open(adapter_config_path, "w", encoding="utf-8") as f:
    json.dump(adapter_config, f, ensure_ascii=False, indent=2)
print("حُفظ LoRA Adapter المتوافق مبدئيًا مع Cloudflare في:", DRIVE_OUTPUT_DIR)
print("الملفات المطلوبة للرفع من واجهة Zazo: adapter_config.json وadapter_model.safetensors")


## 11. تجربة توليد بسيطة (اختيارية)

In [ ]:
model.eval()
test_messages = [{"role": "user", "content": "كيف يربط القرآن بين التدبر والوعي؟ أجب مع ذكر مرجع الآية."}]
prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=160, do_sample=True, temperature=0.7, top_p=0.9)
print(tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))


## ملاحظات تشغيل مهمة

- إذا ظهر CAPTCHA، أكمله يدويًا في Colab؛ لا يمكن للدفتر تجاوزه.
- إذا كان `torch.cuda.is_available()` يساوي `False` أو لم يعمل `nvidia-smi`، فلن يبدأ التدريب حتى تخصص Colab GPU للجلسة.
- حفظ Drive يتم للحساب الذي تختاره في نافذة Google عند `drive.mount`؛ افحص البريد قبل منح الإذن.
- هذه العملية تحفظ محول LoRA فقط، ولا تدمجه تلقائيًا في أوزان النموذج الأساسي ولا تنشره إلى Cloudflare.
- بيانات التدريب عن الآيات صياغات تعليمية أولية مع أرقام مراجع؛ راجعها علميًا/شرعيًا قبل اعتمادها تدريبًا نهائيًا.
